# Visualize the DCRNN Traffic-Signal Graph

Use this notebook to build the same graph topology used by the DCRNN-based experiments and visualize it on top of the SUMO road network.

## What this notebook does

1. Loads one scenario config from `configs/scenario/`.
2. Builds the graph observation wrapper used by the DCRNN path.
3. Extracts the adjacency matrix and directed edge list.
4. Draws the graph over the SUMO map and shows the adjacency heatmap.

This notebook uses the repo's current `build_traffic_signal_graph(...)` logic, so the picture should match the DCRNN topology actually seen by training.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sumolib
from matplotlib.patches import FancyArrowPatch
from omegaconf import OmegaConf

try:
    import pandas as pd
except ImportError:
    pd = None


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "sumo_rl").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sumo_rl.environment.graph_env import build_graph_parallel_env, resolve_sumo_base_env

plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.grid"] = False

print(f"Repo root: {ROOT}")
print(f"pandas available: {pd is not None}")

In [ ]:
SCENARIO_NAME = "resco_ingolstadt21"
HISTORY_LEN = 5
INCLUDE_VIRTUAL_NODES = True
ADD_SELF_LOOPS = True
SHOW_SELF_LOOPS = False
SAVE_FIGURES = False
OUTPUT_DIR = ROOT / "experiments" / "artifacts" / "graph_viz"

SCENARIO_NAME

In [ ]:
def load_cfg(scenario_name: str):
    base_cfg = OmegaConf.load(ROOT / "configs" / "base.yaml")
    scenario_cfg = OmegaConf.load(ROOT / "configs" / "scenario" / f"{scenario_name}.yaml")
    cfg = OmegaConf.merge(base_cfg, scenario_cfg)
    cfg.experiment.name = f"notebook_{scenario_name}_graph"
    return cfg


def resolve_repo_path(raw_path: str | Path) -> Path:
    path = Path(str(raw_path))
    if path.is_absolute():
        return path
    text = str(raw_path).replace("\\", "/")
    if text.startswith("sumo_rl/"):
        return ROOT / text
    return ROOT / path


def lane_shape_points(net, lane_ids):
    points = []
    for lane_id in lane_ids:
        try:
            lane = net.getLane(str(lane_id))
            points.extend((float(x), float(y)) for x, y in lane.getShape())
        except Exception:
            continue
    return points


def mean_point(points):
    if not points:
        return None
    return (
        sum(point[0] for point in points) / len(points),
        sum(point[1] for point in points) / len(points),
    )


def traffic_signal_positions(net, base_env, ts_ids):
    positions = {}
    traffic_signals = getattr(base_env, "traffic_signals", {}) or {}
    for ts_id in ts_ids:
        try:
            x, y = net.getNode(str(ts_id)).getCoord()
            positions[str(ts_id)] = (float(x), float(y))
            continue
        except Exception:
            pass

        signal = traffic_signals.get(str(ts_id))
        if signal is None:
            continue
        lane_ids = list(getattr(signal, "lanes", []) or []) + list(getattr(signal, "out_lanes", []) or [])
        center = mean_point(lane_shape_points(net, lane_ids))
        if center is not None:
            positions[str(ts_id)] = center
    return positions


def edge_shapes(net):
    shapes = []
    for edge in net.getEdges():
        try:
            shape = [(float(x), float(y)) for x, y in edge.getShape()]
        except Exception:
            shape = []
        if len(shape) >= 2:
            shapes.append(shape)
    return shapes


def add_virtual_node_positions(graph, positions):
    real_points = [positions[ts_id] for ts_id in graph.ts_ids if ts_id in positions]
    if not real_points:
        return positions
    min_x = min(point[0] for point in real_points)
    max_x = max(point[0] for point in real_points)
    center_y = sum(point[1] for point in real_points) / len(real_points)
    span_x = max(max_x - min_x, 1.0)
    if graph.incoming_node_index is not None:
        positions["__incoming__"] = (min_x - 0.20 * span_x, center_y)
    if graph.outgoing_node_index is not None:
        positions["__outgoing__"] = (max_x + 0.20 * span_x, center_y)
    return positions


def node_labels(graph):
    labels = list(graph.ts_ids)
    if graph.incoming_node_index is not None:
        labels.append("IN")
    if graph.outgoing_node_index is not None:
        labels.append("OUT")
    return labels


def node_display_labels(graph):
    labels = []
    for index, ts_id in enumerate(graph.ts_ids, start=1):
        if str(ts_id).startswith("cluster_"):
            labels.append(f"TS{index}")
        else:
            labels.append(str(ts_id))
    if graph.incoming_node_index is not None:
        labels.append("IN")
    if graph.outgoing_node_index is not None:
        labels.append("OUT")
    return labels


def node_label_rows(graph):
    raw_labels = node_labels(graph)
    display_labels = node_display_labels(graph)
    rows = []
    for index, (raw_label, display_label) in enumerate(zip(raw_labels, display_labels)):
        rows.append(
            {
                "node_index": index,
                "display_label": display_label,
                "node_id": raw_label,
            }
        )
    return rows


def graph_edge_rows(graph):
    labels = node_display_labels(graph)
    rows = []
    for source, target in np.argwhere(graph.adjacency > 0):
        rows.append(
            {
                "source_index": int(source),
                "target_index": int(target),
                "source": labels[int(source)],
                "target": labels[int(target)],
                "is_self_loop": bool(source == target),
            }
        )
    return rows

In [ ]:
try:
    graph_env.close()
except Exception:
    pass

cfg = load_cfg(SCENARIO_NAME)
params = {
    "history_len": HISTORY_LEN,
    "include_virtual_nodes": INCLUDE_VIRTUAL_NODES,
    "add_self_loops": ADD_SELF_LOOPS,
}
graph_env = build_graph_parallel_env(
    cfg,
    ROOT / "outputs" / "_notebooks",
    seed=int(cfg.experiment.seed),
    params=params,
    use_libsumo=False,
)
base_env = resolve_sumo_base_env(graph_env)
graph = graph_env.graph
net_file = resolve_repo_path(cfg.env.kwargs.net_file)
net = sumolib.net.readNet(str(net_file))
road_shapes = edge_shapes(net)
positions = add_virtual_node_positions(
    graph,
    traffic_signal_positions(net, base_env, list(graph.ts_ids)),
)
edge_rows = graph_edge_rows(graph)
label_rows = node_label_rows(graph)

print(f"Scenario: {SCENARIO_NAME}")
print(f"Net file: {net_file}")
print(f"Traffic-signal nodes: {len(graph.ts_ids)}")
print(f"Graph nodes total: {graph.num_nodes}")
print(f"Directed edges (including self loops): {int(graph.adjacency.sum())}")
print(f"Feature dim per node: {graph.feature_dim}")

In [ ]:
labels = node_display_labels(graph)
index_to_point = {}
for ts_id, index in graph.ts_index.items():
    if ts_id in positions:
        index_to_point[int(index)] = positions[ts_id]
if graph.incoming_node_index is not None and "__incoming__" in positions:
    index_to_point[int(graph.incoming_node_index)] = positions["__incoming__"]
if graph.outgoing_node_index is not None and "__outgoing__" in positions:
    index_to_point[int(graph.outgoing_node_index)] = positions["__outgoing__"]

fig_map, ax_map = plt.subplots(figsize=(10, 8))

for shape in road_shapes:
    xs = [point[0] for point in shape]
    ys = [point[1] for point in shape]
    ax_map.plot(xs, ys, color="#94a3b8", linewidth=1.0, alpha=0.45, zorder=1)

for source, target in np.argwhere(graph.adjacency > 0):
    source = int(source)
    target = int(target)
    if source == target and not SHOW_SELF_LOOPS:
        continue
    if source not in index_to_point or target not in index_to_point:
        continue
    start = index_to_point[source]
    end = index_to_point[target]
    arrow = FancyArrowPatch(
        start,
        end,
        arrowstyle="->",
        mutation_scale=10,
        linewidth=1.4,
        color="#dc2626",
        alpha=0.80,
        shrinkA=10,
        shrinkB=10,
        zorder=2,
        connectionstyle="arc3,rad=0.08" if source != target else "arc3,rad=0.35",
    )
    ax_map.add_patch(arrow)

for index, label in enumerate(labels):
    if index not in index_to_point:
        continue
    x, y = index_to_point[index]
    is_virtual = label in {"IN", "OUT"}
    face = "#0f766e" if not is_virtual else "#7c3aed"
    size = 42 if not is_virtual else 56
    ax_map.scatter([x], [y], s=size, color=face, edgecolors="white", linewidths=0.9, zorder=3)
    ax_map.text(x + 4, y + 4, label, fontsize=8, color="#0f172a", zorder=4)

ax_map.set_title(f"DCRNN topology on SUMO map: {SCENARIO_NAME}")
ax_map.set_aspect("equal", adjustable="box")
ax_map.axis("off")

fig_map.tight_layout()

fig_adj, ax_adj = plt.subplots(figsize=(8, 7))
image = ax_adj.imshow(graph.adjacency, cmap="Blues", interpolation="nearest")
ax_adj.set_title("Adjacency matrix")
ax_adj.set_xticks(range(len(labels)))
ax_adj.set_yticks(range(len(labels)))
ax_adj.set_xticklabels(labels, rotation=90)
ax_adj.set_yticklabels(labels)
fig_adj.colorbar(image, ax=ax_adj, fraction=0.046, pad=0.04)

fig_adj.tight_layout()

if SAVE_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    map_stem = f"{SCENARIO_NAME}_dcrnn_graph_map"
    adj_stem = f"{SCENARIO_NAME}_dcrnn_graph_adjacency"
    fig_map.savefig(OUTPUT_DIR / f"{map_stem}.png", bbox_inches="tight")
    fig_adj.savefig(OUTPUT_DIR / f"{adj_stem}.png", bbox_inches="tight")
    print(f"Saved figures under {OUTPUT_DIR}")

plt.figure(fig_map.number)
plt.show()
plt.figure(fig_adj.number)
plt.show()

In [ ]:
if pd is not None:
    display(pd.DataFrame(label_rows))
    pd.DataFrame(edge_rows)
else:
    {"labels": label_rows[:10], "edges": edge_rows[:10]}

In [ ]:
# Run this cell when you are done with the notebook session.
graph_env.close()